# Lưu cache relation frames v2 — Output105 thành Dataset
**CPU / Accelerator=None / Internet ON / Save & Run All. Không GPU, không chạy lại chuẩn bị.**
Tải đúng version105 của `ducanh2403/2-vi-cai-kia-qua-nhieu`, kiểm hash cả5file gốc,
gồm `inputs.h5`, rồi tạo Dataset Private mới. Không tải latest hoặc tự dùng version khác.
Không cần Add Input hoặc checkpoint. Cần khoảng2GiB trống cho bản tải và bản sao.
Thêm/bật Kaggle Secret `KAGGLE_API_TOKEN`. Nếu secret của bạn tên `codex`, sửa SECRET_NAME.
Tên secret khác tên gợi nhớ của token trên trang Settings. Không dán token vào code hoặc chat.
OWNER phải khớp tài khoản có quyền xem output105 và tạo Dataset.


In [ ]:
from pathlib import Path
KERNEL_VERSION='ducanh2403/2-vi-cai-kia-qua-nhieu/105'
OWNER='ducanh2403'
SLUG='relation-frames-v2-cache-v105'
TITLE='Relation frames v2 cache v105 dataset'
SECRET_NAME='KAGGLE_API_TOKEN'
DESTINATION=Path('/kaggle/working/relation_frames_v2_dataset')


In [ ]:
import os,re,subprocess,sys
from kaggle_secrets import UserSecretsClient
if KERNEL_VERSION!='ducanh2403/2-vi-cai-kia-qua-nhieu/105':raise ValueError('This notebook pins105; do not substitute latest or another version.')
if not re.fullmatch(r'[a-zA-Z0-9_-]+',OWNER) or not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*',SLUG):raise ValueError('Invalid OWNER/SLUG')
if not 6<=len(SLUG)<=50 or not 6<=len(TITLE)<=50:raise ValueError('TITLE/SLUG must have6–50characters')
try:
    token=UserSecretsClient().get_secret(SECRET_NAME).strip()
    if not token:raise ValueError('empty')
except Exception:raise RuntimeError('Enable the configured SECRET_NAME in Add-ons > Secrets. Do not paste token in code/chat.') from None
def run_cli(args,authenticated=True):
    environment=os.environ.copy()
    for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):environment.pop(key,None)
    if authenticated:environment['KAGGLE_API_TOKEN']=token
    environment['PYTHONUNBUFFERED']='1'
    with subprocess.Popen(args,env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace') as process:
        for line in process.stdout:print(line.replace(token,'[REDACTED]'),end='',flush=True)
        code=process.wait()
    if code:raise RuntimeError(f'Command failed ({code}); see redacted output. No automatic fallback or dataset overwrite.')

run_cli([sys.executable,'-m','pip','install','--upgrade','--force-reinstall','--quiet','https://github.com/Kaggle/kaggle-cli/archive/6894864ff4095cf98e9e36d3a694ff5fc9ba3818.zip'],authenticated=False)
VERSION_PREFLIGHT="import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\n\n# Exercise the installed download implementation without contacting Kaggle.\napi = KaggleApi()\nwith tempfile.TemporaryDirectory() as directory, patch.object(api, 'build_kaggle_client') as factory:\n    endpoint = factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value = SimpleNamespace(files=[], next_page_token=None, log=None)\n    api.kernels_output('ducanh2403/2-vi-cai-kia-qua-nhieu/105', path=directory)\n    request = endpoint.call_args.args[0]\n    assert getattr(request, 'version_label', None) == 'v105', 'Installed CLI ignores the requested version; STOP before download.'\n    assert request.user_name == 'ducanh2403'\n    assert request.kernel_slug == '2-vi-cai-kia-qua-nhieu'\nprint('Version routing preflight PASS: request.version_label=v105; no network call.', flush=True)\n"
run_cli([sys.executable,'-u','-c',VERSION_PREFLIGHT])


In [ ]:
import hashlib,json
INVENTORY={'schema': 'relation_frames_v2_transport_v1', 'kernel_version': 'ducanh2403/2-vi-cai-kia-qua-nhieu/105', 'source_review_zip_sha256': '846d4fb94125aa39d6a2825d14a6489c5f60a58e50ae92b30c5cce79fa50eb91', 'files': {'inputs.h5': 'daf32d52859452eb2beb8c35e2aad6ed6418cf52c68e4ba646be633151a7a1e9', 'relation_frames_manifest.bin': '6ca6af23d0a31043fecdf33398f5b60098be07cbeea9fb91cbfec4b314c2cfa7', 'images.json': '531ac7ef4d794652a738af572f157f06c44f9b4a57ecb9926ebfa4fb29aa8923', 'detector_boxes.json': '57bf20b276f937644cf01751e619add484b44e08711811f829633c6be125c677', 'receipt.json': '9bc63a85ce59084d029739d55baec4d2ca359c3afeb78ff320a22b1f310f5915'}}
SOURCE_CODE='"""Verify/copy exact version105 cache bytes; standard library only, no API/model."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\n\nNAMES={\'receipt.json\',\'inputs.h5\',\'relation_frames_manifest.bin\',\'images.json\',\'detector_boxes.json\'}\n\n\ndef sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda:f.read(8*1024**2),b\'\'):h.update(b)\n    return h.hexdigest()\n\n\ndef package(source,destination,inventory):\n    expected=inventory[\'files\']\n    if inventory[\'schema\']!=\'relation_frames_v2_transport_v1\' or set(expected)!=NAMES:raise ValueError(\'Wrong inventory\')\n    if any(not isinstance(h,str) or not re.fullmatch(\'[0-9a-f]{64}\',h) for h in expected.values()):raise ValueError(\'Invalid pinned hashes\')\n    source=Path(source).resolve();destination=Path(destination).resolve()\n    if source==destination or source in destination.parents or destination in source.parents:raise ValueError(\'Use separate non-nested folders\')\n    for n,h in expected.items():\n        p=source/n\n        if p.is_symlink() or not p.is_file() or sha(p)!=h:raise ValueError(\'Missing/changed source: \'+n)\n        print(\'Verified source:\',n,flush=True)\n    receipt=json.loads((source/\'receipt.json\').read_text(encoding=\'utf-8\'))\n    if receipt[\'status\']!=\'CACHE_COPY_COMPLETE_PENDING_LABEL_AUDIT\' or receipt[\'images\']!=612 or receipt[\'legacy_prompt_coverage\']!=612:\n        raise ValueError(\'Wrong completed cache\')\n    if any(receipt[k] for k in (\'feature_extraction\',\'caption_training\',\'neural_forward\')):raise ValueError(\'Unexpected model execution\')\n    if receipt[\'payload_sha256\']!={n:h for n,h in expected.items() if n!=\'receipt.json\'}:raise ValueError(\'Receipt manifest differs\')\n    destination.mkdir(parents=True,exist_ok=True)\n    allowed=NAMES|{n+\'.partial\' for n in NAMES}|{\'transport_receipt.json\'}\n    if any(p.is_symlink() or p.name not in allowed for p in destination.iterdir()):raise ValueError(\'Unrelated destination preserved\')\n    needed=sum((source/n).stat().st_size for n in expected if not (destination/n).exists())\n    if shutil.disk_usage(destination).free<needed+256*1024**2:raise RuntimeError(\'Insufficient copy space\')\n    for n,h in expected.items():\n        target=destination/n\n        if target.exists():\n            if not target.is_file() or sha(target)!=h:raise ValueError(\'Changed destination preserved: \'+n)\n            continue\n        partial=destination/(n+\'.partial\');shutil.copyfile(source/n,partial)\n        if sha(partial)!=h:raise ValueError(\'Copy differs: \'+n)\n        partial.replace(target)\n    if any(p.name not in NAMES|{\'transport_receipt.json\'} for p in destination.iterdir()):raise ValueError(\'Partial files remain\')\n    report=dict(status=\'DATASET_FOLDER_READY\',schema=inventory[\'schema\'],files=expected,payload_files=5,\n        bytes=sum((source/n).stat().st_size for n in expected),source_kernel_version=inventory[\'kernel_version\'],\n        source_review_zip_sha256=inventory[\'source_review_zip_sha256\'],neural_forward=False,caption_training=False,\n        note=\'Five original files unchanged at dataset root; full HDF5 hash verified during transport.\')\n    (destination/\'transport_receipt.json\').write_text(json.dumps(report,indent=2)+\'\\n\',encoding=\'utf-8\')\n    return report\n'
assert hashlib.sha256(SOURCE_CODE.encode()).hexdigest()=='8d469cf97858d38b93267fc784ed576742faae755227faced594099cce0a00d7'
exec(compile(SOURCE_CODE,'package_frames_dataset.py','exec'))


In [ ]:
import tempfile,shutil
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<2*1024**3:
    raise RuntimeError('Need about2GiB free for download+copy. Preserve existing outputs.')
DOWNLOAD=Path(tempfile.mkdtemp(prefix='relation-frames-v105-',dir='/tmp'))
pattern=r'(^|/)(?:'+'|'.join(re.escape(n) for n in INVENTORY['files'])+r')$'
print('Download EXACT source:',KERNEL_VERSION,flush=True)
run_cli([sys.executable,'-m','kaggle','kernels','output',KERNEL_VERSION,'-p',str(DOWNLOAD),'--file-pattern',pattern])
matches=[p.parent for p in DOWNLOAD.rglob('receipt.json') if sha(p)==INVENTORY['files']['receipt.json']]
if len(matches)!=1:raise RuntimeError(f'Output105 returned {len(matches)} matching receipts. STOP; no latest/version fallback.')
report=package(matches[0],DESTINATION,INVENTORY)
print('All5original files and full HDF5 verified; dataset staging ready:',DESTINATION,flush=True)


In [ ]:
CREATE_DATASET_SCRIPT="import sys\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi = KaggleApi()\napi.authenticate()\nresult = api.dataset_create_new(sys.argv[1], public=False, quiet=False,\n                                convert_to_csv=False, dir_mode='skip')\nif result is None or str(getattr(result, 'status', '')).lower() != 'ok' or getattr(result, 'error', None):\n    raise RuntimeError('Dataset creation rejected: ' + str(getattr(result, 'error', 'Missing success response')))\nprint('Dataset creation accepted by API:', result.url, flush=True)\n"

metadata_path=DESTINATION/'dataset-metadata.json'
metadata=dict(title=TITLE,id=f'{OWNER}/{SLUG}',licenses=[dict(name='other')],
    description='Private research cache from exact notebook Output105. Original COCO/CLIP terms retained. Copies existing visual and original prompt embeddings for612images; paired VSR metadata and predicted boxes. No new feature extraction, trained checkpoint or model-quality claim. Source and transport receipts preserved.')
if metadata_path.exists():raise RuntimeError('Existing metadata preserved; use fresh staging destination.')
try:
    metadata_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    run_cli([sys.executable,'-u','-c',CREATE_DATASET_SCRIPT,str(DESTINATION)])
finally:metadata_path.unlink(missing_ok=True)
print('API accepted creation; processing may still be pending. Do not repeatedly create.',flush=True)
print(f'https://www.kaggle.com/datasets/{OWNER}/{SLUG}',flush=True)
try:run_cli([sys.executable,'-m','kaggle','datasets','status',f'{OWNER}/{SLUG}'])
except RuntimeError:print('Status not confirmed. Check dataset page before retrying.',flush=True)
try:run_cli([sys.executable,'-m','kaggle','datasets','files',f'{OWNER}/{SLUG}','--page-size','100'])
except RuntimeError:print('File listing unavailable yet; wait for processing, do not rerun create.',flush=True)


Sau khi xử lý xong, Dataset có5file gốc ở ngay thư mục gốc:
`inputs.h5`, `relation_frames_manifest.bin`, `images.json`, `detector_boxes.json`, `receipt.json`,
cộng `transport_receipt.json`. Không có folderImage_Captioning hoặc hàng trăm file source.
`.bin` là metadata nén nguyên byte, không phải checkpoint; không cần tự giải nén.
Nếu dùng SLUG mặc định, path dự kiến:
`/kaggle/input/datasets/ducanh2403/relation-frames-v2-cache-v105`.
Gửi path mount thực tế khi attach dataset. Chia sẻ Private với Ngân nếu dùng tài khoản thứ hai.
Không ghi đè Dataset đã có; nếu slug tồn tại, kiểm tra trang trước khi tạo lại.
Cờ audit=false trong receipt gốc là trạng thái tại lúc copy; review32mẫu đã hoàn tất được lưu riêng.
Notebook này chưa được Codex gọi API hoặc chạy trên Kaggle.
